# Demo: One-class SVM with Probability

**Bai bao**: Que & Lin, "One-Class SVM Probabilistic Outputs", IEEE TNNLS, thang 4/2025.

**Van de**: One-class SVM chi tra ra mot diem so (decision value), khong phai xac suat. Bai bao chi ra rang phuong phap Platt scaling (chuan cho SVM 2 lop) khong dung duoc o day vi khong co nhan that. Bai bao de xuat 2 phuong phap thay the:

1. **Binning** theo decision value (phien ban density da duoc tich hop vao LIBSVM tu phien ban 3.3)
2. **New Gamma scaling** (phuong phap tham so, dung phan phoi Gamma)

**Demo nay** so sanh ca 4 phuong phap (Platt, equidistant binning, density binning, Gamma scaling) tren bo du lieu **breast cancer** cua sklearn. Y tuong: train one-class SVM chi tren mau lanh tinh, sau do test xem khi co mau ac tinh thi mo hinh canh bao the nao, va xac suat tra ra co chinh xac (calibrated) khong.

Notebook chay doc lap khong can mang. Du lieu di kem sklearn.

In [ ]:
# cac thu vien can thiet
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import gamma
from sklearn.datasets import load_breast_cancer
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# seed de chay lai cho ket qua giong nhau
rng = np.random.default_rng(42)

## 1. Chuan bi du lieu

Bo `breast_cancer` cua sklearn co 569 mau, 30 dac trung, nhan nhi phan (1 = lanh tinh, 0 = ac tinh).

Quy uoc one-class SVM: chi cho mo hinh nhin thay mau "binh thuong" (lanh tinh) trong luc train. Mau ac tinh dong vai tro "bat thuong" va chi xuat hien o tap test. Day la kich ban thuc te cua bai toan phat hien bat thuong trong y te: ta co nhieu vi du benh nhan khoe manh, rat it vi du bat thuong, va phai canh bao khi gap bat thuong.

In [ ]:
# load du lieu
bc = load_breast_cancer()
Xall, yall = bc.data, bc.target

# tach chi so theo nhan
normal_idx = np.where(yall == 1)[0]    # lanh tinh
anomaly_idx = np.where(yall == 0)[0]   # ac tinh
rng.shuffle(normal_idx)

# 60% mau lanh tinh dem train, con lai gop voi tat ca mau ac tinh thanh tap test
cut = int(len(normal_idx) * 0.6)
fit_idx = normal_idx[:cut]
held_normal = normal_idx[cut:]

# chuan hoa dac trung: scaler chi fit tren tap train
sc = StandardScaler().fit(Xall[fit_idx])
Xfit = sc.transform(Xall[fit_idx])
Xeval = sc.transform(np.vstack([Xall[held_normal], Xall[anomaly_idx]]))
yeval = np.r_[np.ones(len(held_normal)), np.zeros(len(anomaly_idx))]

print(f"So mau train (lanh tinh): {len(fit_idx)}")
print(f"So mau test:              {len(Xeval)}")
print(f"Ti le lanh tinh trong test: {yeval.mean():.3f}")

## 2. Train one-class SVM va xem decision value

Dung kernel RBF, `nu = 0.1` (cho phep toi da 10% mau train bi coi la bat thuong). Sau khi train, ham `decision_function(x)` tra ve diem so `g(x)`: duong = giong mau lanh tinh, am = bat thuong.

Histogram ben duoi cho thay phan bo diem so cua **mau lanh tinh** (xanh) va **mau ac tinh** (do) tren tap test. Hai phan bo chong lan o vung gan 0 - day chinh la vung ma xac suat thuc su co y nghia, va cung la vung Platt scaling sap that bai.

In [ ]:
svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
svm.fit(Xfit)

g_fit  = svm.decision_function(Xfit)
g_eval = svm.decision_function(Xeval)
fmax = g_fit.max()   # luu lai de dung trong gamma scaling

# ve histogram
plt.figure(figsize=(7, 3))
plt.hist(g_eval[yeval == 1], bins=24, color='#2a6', alpha=0.6, label='lanh tinh')
plt.hist(g_eval[yeval == 0], bins=24, color='#c33', alpha=0.6, label='ac tinh')
plt.axvline(0, color='k', lw=0.7, ls='--')
plt.xlabel('g(x) = decision value')
plt.ylabel('so mau')
plt.legend()
plt.title('Phan bo decision value tren tap test')
plt.tight_layout()
plt.show()

## 3. Bon phuong phap chuyen decision value sang xac suat

| Phuong phap | Loai | Y tuong |
|---|---|---|
| **Platt** | tham so (sigmoid) | Fit `p = sigmoid(A*g + B)` voi nhan gia tu chinh OCSVM. Bai bao chi ra cach nay sup do thanh ham buoc 0/1 vi thieu nhan that. |
| **Equidistant binning** | khong tham so | Chia khoang `[g_min, 0]` va `[0, g_max]` thanh K phan deu nhau. |
| **Density binning** | khong tham so | Chia theo quantile (mat do diem). Day la phuong phap LIBSVM 3.3+ dung mac dinh. |
| **New Gamma scaling** | tham so | Fit phan phoi Gamma cho `S = max(g_max - g, 0)`, scale sao cho `g = 0 -> p = 0.5`. |

Cac cell duoi day cai dat tung phuong phap.

In [ ]:
# Platt scaling (Lin et al 2007, dang on dinh so hoc)
def platt(g_train, g_query, pseudo):
    pseudo = pseudo.astype(float)
    Np = pseudo.sum()
    Nn = len(pseudo) - Np
    # target voi smoothing chong overfit
    t = np.where(pseudo > 0, (Np + 1) / (Np + 2), 1 / (Nn + 2))

    def J(ab):
        z = ab[0] * g_train + ab[1]
        return (t * z + np.log1p(np.exp(-z))).sum()

    out = minimize(J, [0.0, 0.0], method='Nelder-Mead', options={'xatol': 1e-5})
    A, B = out.x
    return expit(-(A * g_query + B))

# nhan gia: svm.predict tra +1 cho mau giong train, -1 cho bat thuong
pseudo = (svm.predict(Xfit) > 0).astype(int)

In [ ]:
# hai phuong phap binning
EPS = 0.001
K = 5  # so thung moi ben g = 0

def _nearest_mark(marks, probs, q):
    # gan moi diem query vao mark gan nhat, lay xac suat tuong ung
    j = np.abs(q[:, None] - marks[None, :]).argmin(axis=1)
    return probs[j]

def equidistant(g_train, g_query):
    # Binning chia khoang deu nhau
    a, b = g_train.min(), g_train.max()
    left  = np.linspace(a, 0, K + 1)[:-1]
    right = np.linspace(0, b, K + 1)[1:]
    marks = np.r_[left, 0.0, right]
    ps = np.linspace(EPS, 1 - EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

def density_bin(g_train, g_query):
    # Binning chia theo quantile (LIBSVM default)
    qs = (np.arange(K) + 0.5) / K
    neg = g_train[g_train < 0]
    pos = g_train[g_train >= 0]
    nm = np.quantile(neg, qs) if len(neg) else np.zeros(K)
    pm = np.quantile(pos, qs) if len(pos) else np.zeros(K)
    marks = np.r_[nm, 0.0, pm]
    ps = np.linspace(EPS, 1 - EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

In [ ]:
# New Gamma scaling - phuong phap moi cua bai bao (cong thuc 40)
def gamma_scale(g_train, g_query):
    # buoc 1: tinh S = (fmax - g)+ tren tap train
    S = np.clip(fmax - g_train, 0, None)
    S = S[S > 0]

    # buoc 2: fit Gamma bang phuong phap moment
    # (Satterthwaite-Welch xap xi tong chi-binh-phuong)
    m, v = S.mean(), S.var()
    shape, scale = m * m / v, v / m

    # buoc 3: tinh survival function
    Sq = np.clip(fmax - g_query, 0, None)
    surv = 1 - gamma.cdf(Sq, shape, scale=scale)
    surv0 = 1 - gamma.cdf(fmax, shape, scale=scale)
    surv0 = max(surv0, 1e-9)  # tranh chia 0

    # buoc 4: scale theo cong thuc (40) sao cho g=0 -> p=0.5
    up = 0.5 + 0.5 * (surv - surv0) / max(1 - surv0, 1e-9)
    dn = 0.5 * surv / surv0
    return np.where(surv >= surv0, up, dn).clip(EPS, 1 - EPS)

## 4. Ket qua: 3 benh nhan ngau nhien

Tinh xac suat lanh tinh theo ca 4 phuong phap, in ra cho 3 benh nhan duoc chon ngau nhien tu tap test. Cot `y` la nhan that (1 = lanh tinh, 0 = ac tinh).

**Diem can de y**: Platt thuong cho ra gia tri rat gan 1 hoac rat gan 0 (gan nhu nhi phan), trong khi Gamma scaling cho ra gia tri co do tin cay khac nhau ro rang.

In [ ]:
# tinh xac suat theo ca 4 phuong phap
probs = {
    'platt':       platt(g_fit, g_eval, pseudo),
    'equi-bin':    equidistant(g_fit, g_eval),
    'dens-bin':    density_bin(g_fit, g_eval),
    'gamma-scale': gamma_scale(g_fit, g_eval),
}

# chon 3 benh nhan ngau nhien
trio = rng.choice(len(g_eval), size=3, replace=False)

header = f"{'row':>4}  {'g':>7}  {'y':>2}    " + "   ".join(f"{k:>11}" for k in probs)
print(header)
print("-" * len(header))
for r in trio:
    row = f"{r:>4}  {g_eval[r]:+.3f}  {int(yeval[r]):>2}    "
    row += "   ".join(f"{probs[k][r]:>11.3f}" for k in probs)
    print(row)

## 5. Reliability plot (do calibration)

Chia tap test thanh 10 thung theo quantile cua xac suat du doan. Voi moi thung, lay xac suat trung binh va ti le lanh tinh thuc te. Mot mo hinh **calibrated tot** se nam tren duong cheo `y = x`: "neu mo hinh noi 80% lanh tinh thi 80/100 truong hop nhu the la lanh tinh that".

Duong cheo xam la muc tieu hoan hao. Duong nao gan no la phuong phap tot.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], color='#999', lw=0.8, ls='--', label='hoan hao (y=x)')

for name, p in probs.items():
    # 10 thung quantile
    e = np.quantile(p, np.linspace(0, 1, 11))
    e[0] -= 1e-6
    e[-1] += 1e-6
    xs, ys = [], []
    for i in range(10):
        m = (p >= e[i]) & (p < e[i + 1])
        if m.sum() >= 2:
            xs.append(p[m].mean())
            ys.append(yeval[m].mean())
    ax.plot(xs, ys, '.-', label=name, lw=1.2, markersize=8)

ax.set_xlabel('xac suat du doan (mean trong thung)')
ax.set_ylabel('ti le lanh tinh thuc te')
ax.set_title('Reliability plot')
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Brier score

Brier score = MSE giua xac suat du doan va nhan that. **Cang nho cang tot**. Day la chi so tong hop calibration cho ca tap test.

In [ ]:
print(f"{'phuong phap':<14} Brier score")
print("-" * 30)
for name, p in probs.items():
    brier = np.mean((p - yeval) ** 2)
    print(f"  {name:<12} {brier:.4f}")

## Nhan xet

- **Platt scaling** sup do gan ve ham buoc 0/1 nhu bai bao da du bao (cell 3 benh nhan o tren, gia tri Platt thuong rat gan 1 hoac rat gan 0).
- **Equidistant binning** cho ket qua tho, bi anh huong neu co outlier keo dai khoang `[g_min, g_max]`.
- **Density binning** on dinh hon, theo sat duong cheo trong reliability plot.
- **New Gamma scaling** cho xac suat co do tin cay phan loai ro hon va Brier score thuong la thap nhat.

Demo nay xac nhan ket luan cua Que & Lin 2025 tren bo du lieu thuc te (khong phai ART1 toy data trong bai bao).